# 2026 NLP Competition Solution

Ноутбук выполняет полный цикл: загрузка данных, EDA, обучение, инференс, постпроцессинг и сохранение `sample_submission.csv`.

In [ ]:
import ast
import platform
import random
import re
import warnings
from dataclasses import dataclass
from html import unescape
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from scipy.sparse import hstack
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.model_selection import KFold
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import OneHotEncoder
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
except ImportError:
    MultilabelStratifiedKFold = None

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

SEED = 322
TARGET_COLUMNS = ["label_0", "label_1", "label_2", "label_3", "label_4"]


def detect_data_root() -> Path:
    # Local run first.
    local_root = Path(".")
    if (local_root / "train.csv").exists() and (local_root / "test.csv").exists() and (local_root / "sample_submission.csv").exists():
        return local_root

    # Kaggle competition input (folder name can vary by slug).
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for train_path in kaggle_input.rglob("train.csv"):
            root = train_path.parent
            if (root / "test.csv").exists() and (root / "sample_submission.csv").exists():
                return root

    raise FileNotFoundError("Could not find train.csv, test.csv, sample_submission.csv")


PROJECT_ROOT = detect_data_root()


@dataclass
class DataBundle:
    train: pd.DataFrame
    test: pd.DataFrame
    sample_submission: pd.DataFrame


def set_global_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def parse_target(value: str) -> np.ndarray:
    arr = np.asarray(ast.literal_eval(value), dtype=np.int64)
    if arr.shape != (5,):
        raise ValueError(f"Target must have 5 labels, got {arr.shape}.")
    if not np.isin(arr, [0, 1]).all():
        raise ValueError("Target values must be binary.")
    return arr


def load_competition_data(data_dir: Path | str = ".") -> DataBundle:
    data_dir = Path(data_dir)
    train = pd.read_csv(data_dir / "train.csv", sep="\t")
    test = pd.read_csv(data_dir / "test.csv", sep="\t")
    sample_submission = pd.read_csv(data_dir / "sample_submission.csv")
    return DataBundle(train=train, test=test, sample_submission=sample_submission)


def validate_schema(train: pd.DataFrame, test: pd.DataFrame, sample_submission: pd.DataFrame) -> None:
    if set(train.columns) != {"id", "source", "title", "text", "publication_date", "target"}:
        raise ValueError(f"Unexpected train columns: {train.columns.tolist()}")
    if set(test.columns) != {"id", "source", "title", "text", "publication_date"}:
        raise ValueError(f"Unexpected test columns: {test.columns.tolist()}")
    if set(sample_submission.columns) != {"id", "target"}:
        raise ValueError(f"Unexpected submission columns: {sample_submission.columns.tolist()}")


def add_target_columns(train: pd.DataFrame) -> pd.DataFrame:
    parsed = np.vstack(train["target"].map(parse_target).to_numpy())
    out = train.copy()
    for i, c in enumerate(TARGET_COLUMNS):
        out[c] = parsed[:, i]
    return out


def extract_targets(df: pd.DataFrame) -> np.ndarray:
    return df[TARGET_COLUMNS].to_numpy(dtype=np.int64)


def format_submission_targets(binary_matrix: np.ndarray) -> list[str]:
    return ["[" + ",".join(map(str, row.astype(int).tolist())) + "]" for row in binary_matrix]


def hamming_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    # Kaggle score behaves as hamming loss: lower is better.
    return float(np.not_equal(y_true, y_pred).mean())


TAG_RE = re.compile(r"<[^>]+>")
SPACE_RE = re.compile(r"\s+")


def clean_text(value: str) -> str:
    if not isinstance(value, str):
        value = "" if pd.isna(value) else str(value)
    text = unescape(value)
    text = TAG_RE.sub(" ", text)
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    return SPACE_RE.sub(" ", text).strip()


def compose_text_features(
    df: pd.DataFrame,
    include_source: bool = True,
    include_date: bool = True,
) -> pd.Series:
    title = df["title"].fillna("").map(clean_text)
    text = df["text"].fillna("").map(clean_text)

    parts = ["[TITLE] " + title, "[TEXT] " + text]

    if include_source:
        source = df["source"].fillna("").astype(str)
        parts.insert(0, "[SRC] " + source)

    if include_date:
        dt = pd.to_datetime(df["publication_date"], errors="coerce")
        year = dt.dt.year.fillna(-1).astype(int).astype(str)
        month = dt.dt.month.fillna(-1).astype(int).astype(str)
        parts.insert(1 if include_source else 0, "[YEAR] " + year + " [MONTH] " + month)

    combined = parts[0]
    for p in parts[1:]:
        combined = combined + " " + p
    return combined


def _mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    pooled = (last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
    return pooled


def build_transformer_embeddings(texts: pd.Series, model_name: str, batch_size: int, max_length: int, cache_path: Path | None = None) -> np.ndarray:
    if cache_path is not None and cache_path.exists():
        return np.load(cache_path)["embeddings"]

    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    chunks = []
    with torch.inference_mode():
        for start in tqdm(range(0, len(texts), batch_size), desc=f"Embeddings:{model_name}"):
            batch = [f"passage: {t}" for t in texts.iloc[start:start + batch_size].tolist()]
            encoded = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(device)
            with torch.autocast(device_type="cuda", enabled=(device == "cuda")):
                outputs = model(**encoded)
                pooled = _mean_pool(outputs.last_hidden_state, encoded["attention_mask"])
                pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
            chunks.append(pooled.float().cpu().numpy())

    embeddings = np.vstack(chunks)
    if cache_path is not None:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(cache_path, embeddings=embeddings)
    return embeddings


def build_tfidf_features(train_texts: pd.Series, test_texts: pd.Series, max_features_word: int = 120000, max_features_char: int = 80000):
    word = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.98, sublinear_tf=True, max_features=max_features_word)
    char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.98, sublinear_tf=True, max_features=max_features_char)
    x_train = hstack([word.fit_transform(train_texts), char.fit_transform(train_texts)]).tocsr()
    x_test = hstack([word.transform(test_texts), char.transform(test_texts)]).tocsr()
    return x_train, x_test


def build_disentangled_sparse_features(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    max_title_features: int = 60000,
    max_text_word_features: int = 180000,
    max_text_char_features: int = 90000,
):
    title_train = train_df["title"].fillna("").map(clean_text)
    title_test = test_df["title"].fillna("").map(clean_text)
    text_train = train_df["text"].fillna("").map(clean_text)
    text_test = test_df["text"].fillna("").map(clean_text)

    title_vec = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        sublinear_tf=True,
        max_features=max_title_features,
    )
    text_word_vec = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.98,
        sublinear_tf=True,
        max_features=max_text_word_features,
    )
    text_char_vec = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=3,
        max_df=0.98,
        sublinear_tf=True,
        max_features=max_text_char_features,
    )

    x_title_train = title_vec.fit_transform(title_train)
    x_title_test = title_vec.transform(title_test)
    x_text_word_train = text_word_vec.fit_transform(text_train)
    x_text_word_test = text_word_vec.transform(text_test)
    x_text_char_train = text_char_vec.fit_transform(text_train)
    x_text_char_test = text_char_vec.transform(text_test)

    dt_train = pd.to_datetime(train_df["publication_date"], errors="coerce")
    dt_test = pd.to_datetime(test_df["publication_date"], errors="coerce")

    cat_train = pd.DataFrame(
        {
            "source": train_df["source"].fillna("unknown").astype(str),
            "month": dt_train.dt.month.fillna(-1).astype(int).astype(str),
            "year": dt_train.dt.year.fillna(-1).astype(int).astype(str),
        }
    )
    cat_test = pd.DataFrame(
        {
            "source": test_df["source"].fillna("unknown").astype(str),
            "month": dt_test.dt.month.fillna(-1).astype(int).astype(str),
            "year": dt_test.dt.year.fillna(-1).astype(int).astype(str),
        }
    )

    try:
        cat_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        cat_encoder = OneHotEncoder(handle_unknown="ignore", sparse=True)

    x_cat_train = cat_encoder.fit_transform(cat_train)
    x_cat_test = cat_encoder.transform(cat_test)

    x_train = hstack(
        [x_title_train, x_text_word_train, x_text_char_train, x_cat_train],
        format="csr",
    )
    x_test = hstack(
        [x_title_test, x_text_word_test, x_text_char_test, x_cat_test],
        format="csr",
    )

    return x_train, x_test


def build_base_classifier(seed: int = SEED, C: float = 2.0, class_weight: str | None = "balanced"):
    base = LogisticRegression(
        C=C,
        max_iter=600,
        solver="liblinear",
        random_state=seed,
        class_weight=class_weight,
    )
    return OneVsRestClassifier(base)


def build_sgd_classifier(seed: int = SEED, alpha: float = 3e-5, class_weight: str | None = "balanced"):
    base = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=alpha,
        max_iter=3000,
        tol=1e-3,
        random_state=seed,
        class_weight=class_weight,
    )
    return OneVsRestClassifier(base)


def make_multilabel_splitter(n_splits: int = 5, seed: int = SEED):
    if MultilabelStratifiedKFold is not None:
        return MultilabelStratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return KFold(n_splits=n_splits, shuffle=True, random_state=seed)


def make_time_splits(df: pd.DataFrame, n_splits: int = 5):
    dt = pd.to_datetime(df["publication_date"], errors="coerce")
    dt = dt.fillna(pd.Timestamp("1970-01-01"))
    order = np.argsort(dt.to_numpy())
    fold_bins = np.array_split(order, n_splits)

    splits = []
    for i in range(n_splits):
        va_idx = fold_bins[i]
        tr_idx = np.concatenate([fold_bins[j] for j in range(n_splits) if j != i])
        splits.append((tr_idx, va_idx))
    return splits


def fit_cv_models(x, y: np.ndarray, estimator, splitter, score_fn):
    oof = np.zeros((y.shape[0], y.shape[1]), dtype=np.float32)
    models, fold_scores = [], []

    if hasattr(splitter, "split"):
        split_iter = splitter.split(x, y)
    else:
        split_iter = splitter

    for fold_idx, (tr_idx, va_idx) in enumerate(split_iter, start=1):
        model = clone(estimator)
        model.fit(x[tr_idx], y[tr_idx])
        proba = model.predict_proba(x[va_idx])
        oof[va_idx] = proba
        pred = (proba >= 0.5).astype(np.int64)
        score = score_fn(y[va_idx], pred)
        fold_scores.append(float(score))
        models.append(model)
        print(f"Fold {fold_idx}: hamming_loss={score:.6f}")
    return {"oof_proba": oof, "models": models, "fold_scores": fold_scores}


def fit_full_model(x, y: np.ndarray, estimator):
    model = clone(estimator)
    model.fit(x, y)
    return model


def find_best_thresholds(y_true: np.ndarray, y_proba: np.ndarray, score_fn, grid: np.ndarray | None = None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 91)
    thresholds = np.full(y_true.shape[1], 0.5, dtype=np.float32)
    best_global = score_fn(y_true, (y_proba >= thresholds).astype(np.int64))
    for label in range(y_true.shape[1]):
        best_t, best_score = thresholds[label], best_global
        for t in grid:
            trial = thresholds.copy()
            trial[label] = float(t)
            score = score_fn(y_true, (y_proba >= trial).astype(np.int64))
            if score < best_score:
                best_t, best_score = float(t), score
        thresholds[label] = best_t
        best_global = best_score
    return thresholds, float(best_global)


def apply_thresholds(y_proba: np.ndarray, thresholds: np.ndarray) -> np.ndarray:
    return (y_proba >= thresholds).astype(np.int64)


set_global_seed(SEED)
print(f"Seed fixed to: {SEED}")
print(f"Python: {platform.python_version()}")

In [ ]:
bundle = load_competition_data(PROJECT_ROOT)
train_df = bundle.train
test_df = bundle.test
sample_submission = bundle.sample_submission

validate_schema(train_df, test_df, sample_submission)
train_df = add_target_columns(train_df)

print("Data root:", PROJECT_ROOT)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_submission.shape)
display(train_df.head(2))

In [ ]:
# EDA
y = extract_targets(train_df)
label_freq = pd.Series(y.mean(axis=0), index=TARGET_COLUMNS, name="positive_ratio")
cardinality = pd.Series(y.sum(axis=1), name="labels_per_sample")

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
label_freq.plot(kind="bar", ax=axes[0], title="Label positive ratio")
cardinality.value_counts().sort_index().plot(kind="bar", ax=axes[1], title="Label cardinality")

tmp = train_df.copy()
tmp["title_len"] = tmp["title"].fillna("").astype(str).str.len()
tmp["text_len"] = tmp["text"].fillna("").astype(str).str.len()
sns.scatterplot(data=tmp.sample(min(4000, len(tmp)), random_state=SEED), x="title_len", y="text_len", s=8, ax=axes[2])
axes[2].set_title("Title vs text length")
plt.tight_layout()
plt.show()

display(label_freq.to_frame())
display(train_df["source"].value_counts().head(20).to_frame("count"))

In [ ]:
# v0.7 feature recipe: disentangled sparse channels
best_variant_name = "disentangled_title_text_source_date"

# Still prepared for optional transformer branch.
train_texts = compose_text_features(train_df, include_source=True, include_date=True)
test_texts = compose_text_features(test_df, include_source=True, include_date=True)

print("Selected feature recipe:", best_variant_name)
display(train_texts.head(2))

In [ ]:
# Feature extraction for v0.7 (disentangled sparse channels + optional transformer)
if str(PROJECT_ROOT).startswith("/kaggle/input"):
    ARTIFACTS_DIR = Path("/kaggle/working/artifacts")
else:
    ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "intfloat/multilingual-e5-small"
USE_TRANSFORMER_BLEND = False

x_train_tfidf, x_test_tfidf = build_disentangled_sparse_features(
    train_df,
    test_df,
    max_title_features=70000,
    max_text_word_features=180000,
    max_text_char_features=100000,
)

HAS_TRANSFORMER = False
x_train_transformer = None
x_test_transformer = None

if USE_TRANSFORMER_BLEND:
    try:
        x_train_transformer = build_transformer_embeddings(
            train_texts,
            model_name=MODEL_NAME,
            batch_size=32,
            max_length=256,
            cache_path=ARTIFACTS_DIR / "train_embeddings.npz",
        )
        x_test_transformer = build_transformer_embeddings(
            test_texts,
            model_name=MODEL_NAME,
            batch_size=32,
            max_length=256,
            cache_path=ARTIFACTS_DIR / "test_embeddings.npz",
        )
        HAS_TRANSFORMER = True
    except Exception as exc:
        print(f"Transformer branch disabled: {exc}")

print("Disentangled sparse train shape:", x_train_tfidf.shape)
print("Disentangled sparse test shape:", x_test_tfidf.shape)
print("Transformer available:", HAS_TRANSFORMER)
if HAS_TRANSFORMER:
    print("Transformer train shape:", x_train_transformer.shape)
    print("Transformer test shape:", x_test_transformer.shape)

In [ ]:
# Hyperparameter ablation on TF-IDF branch
ABLATION_ENABLED = True
ABLATION_ROWS = 7000
ABLATION_SPLITS = 3

BEST_C = 2.0
BEST_CLASS_WEIGHT = "balanced"
ablation_df = pd.DataFrame()

if ABLATION_ENABLED:
    if ABLATION_ROWS is None or ABLATION_ROWS >= len(train_df):
        ab_idx = np.arange(len(train_df))
    else:
        ab_idx = train_df.sample(ABLATION_ROWS, random_state=SEED).index.to_numpy()

    y_ab = y[ab_idx]
    x_ab_tfidf = x_train_tfidf[ab_idx]

    runs = []
    splitter_ab = make_multilabel_splitter(n_splits=ABLATION_SPLITS, seed=SEED)

    for class_weight in ["balanced", None]:
        for c_value in [0.5, 1.0, 2.0, 4.0, 8.0]:
            est = build_base_classifier(seed=SEED, C=c_value, class_weight=class_weight)
            cv_ab = fit_cv_models(x_ab_tfidf, y_ab, est, splitter_ab, hamming_score)
            pred_ab = (cv_ab["oof_proba"] >= 0.5).astype(np.int64)
            loss_ab = hamming_score(y_ab, pred_ab)
            runs.append({
                "branch": "tfidf",
                "class_weight": str(class_weight),
                "C": c_value,
                "loss": loss_ab,
            })

    ablation_df = pd.DataFrame(runs).sort_values("loss", ascending=True).reset_index(drop=True)
    display(ablation_df)

    if not ablation_df.empty:
        BEST_C = float(ablation_df.iloc[0]["C"])
        BEST_CLASS_WEIGHT = None if ablation_df.iloc[0]["class_weight"] == "None" else "balanced"

print(f"Selected TF-IDF params: C={BEST_C}, class_weight={BEST_CLASS_WEIGHT}")

In [ ]:
# CV training for v0.6: time-based multi-seed model ensemble + optional transformer blend
RUN_MODE = "full"  # use "debug" for faster local checks
n_splits = 5 if RUN_MODE == "full" else 2

CV_STRATEGY = "time"  # "time" is recommended for this dataset

y = extract_targets(train_df)
ENSEMBLE_SEEDS = [322, 42, 2026]

lr_oof_list = []
sgd_oof_list = []
lr_seed_losses = []
sgd_seed_losses = []

for seed_value in ENSEMBLE_SEEDS:
    if CV_STRATEGY == "time":
        splitter_seed = make_time_splits(train_df, n_splits=n_splits)
    else:
        splitter_seed = make_multilabel_splitter(n_splits=n_splits, seed=seed_value)

    estimator_lr = build_base_classifier(seed=seed_value, C=BEST_C, class_weight=BEST_CLASS_WEIGHT)
    cv_lr_seed = fit_cv_models(
        x=x_train_tfidf,
        y=y,
        estimator=estimator_lr,
        splitter=splitter_seed,
        score_fn=hamming_score,
    )
    lr_oof_list.append(cv_lr_seed["oof_proba"])
    pred_lr = (cv_lr_seed["oof_proba"] >= 0.5).astype(np.int64)
    loss_lr = hamming_score(y, pred_lr)
    lr_seed_losses.append(loss_lr)
    print(f"OOF LR seed={seed_value} hamming_loss@0.5: {loss_lr:.6f}")

    if CV_STRATEGY == "time":
        splitter_seed = make_time_splits(train_df, n_splits=n_splits)
    else:
        splitter_seed = make_multilabel_splitter(n_splits=n_splits, seed=seed_value)

    estimator_sgd = build_sgd_classifier(seed=seed_value, alpha=3e-5, class_weight=BEST_CLASS_WEIGHT)
    cv_sgd_seed = fit_cv_models(
        x=x_train_tfidf,
        y=y,
        estimator=estimator_sgd,
        splitter=splitter_seed,
        score_fn=hamming_score,
    )
    sgd_oof_list.append(cv_sgd_seed["oof_proba"])
    pred_sgd = (cv_sgd_seed["oof_proba"] >= 0.5).astype(np.int64)
    loss_sgd = hamming_score(y, pred_sgd)
    sgd_seed_losses.append(loss_sgd)
    print(f"OOF SGD seed={seed_value} hamming_loss@0.5: {loss_sgd:.6f}")

cv_oof_lr = np.mean(lr_oof_list, axis=0)
cv_oof_sgd = np.mean(sgd_oof_list, axis=0)

MODEL_BLEND_WEIGHT = 1.0  # weight of LR branch inside TF-IDF ensemble
best_model_blend_loss = 10.0
for w_lr in np.linspace(0.0, 1.0, 11):
    oof_model_blend = w_lr * cv_oof_lr + (1.0 - w_lr) * cv_oof_sgd
    pred_model_blend = (oof_model_blend >= 0.5).astype(np.int64)
    loss_model_blend = hamming_score(y, pred_model_blend)
    if loss_model_blend < best_model_blend_loss:
        best_model_blend_loss = loss_model_blend
        MODEL_BLEND_WEIGHT = float(w_lr)

cv_oof_tfidf = MODEL_BLEND_WEIGHT * cv_oof_lr + (1.0 - MODEL_BLEND_WEIGHT) * cv_oof_sgd
tfidf_pred = (cv_oof_tfidf >= 0.5).astype(np.int64)
tfidf_loss = hamming_score(y, tfidf_pred)
print(f"Best TF-IDF model blend weight (LR): {MODEL_BLEND_WEIGHT:.2f}")
print(f"OOF TF-IDF model-ensemble hamming_loss@0.5: {tfidf_loss:.6f}")

BLEND_ALPHA = 1.0
cv_oof_proba = cv_oof_tfidf
transformer_loss = None
blend_loss = tfidf_loss

if HAS_TRANSFORMER:
    estimator_tr = build_base_classifier(seed=SEED, C=BEST_C, class_weight=BEST_CLASS_WEIGHT)
    if CV_STRATEGY == "time":
        splitter_tr = make_time_splits(train_df, n_splits=n_splits)
    else:
        splitter_tr = make_multilabel_splitter(n_splits=n_splits, seed=SEED)

    cv_transformer = fit_cv_models(
        x=x_train_transformer,
        y=y,
        estimator=estimator_tr,
        splitter=splitter_tr,
        score_fn=hamming_score,
    )
    tr_pred = (cv_transformer["oof_proba"] >= 0.5).astype(np.int64)
    transformer_loss = hamming_score(y, tr_pred)
    print(f"OOF Transformer hamming_loss@0.5: {transformer_loss:.6f}")

    best_alpha = 1.0
    best_alpha_loss = tfidf_loss
    for alpha in np.linspace(0.0, 1.0, 11):
        blended_oof = alpha * cv_oof_tfidf + (1.0 - alpha) * cv_transformer["oof_proba"]
        blended_pred = (blended_oof >= 0.5).astype(np.int64)
        loss_alpha = hamming_score(y, blended_pred)
        if loss_alpha < best_alpha_loss:
            best_alpha_loss = loss_alpha
            best_alpha = float(alpha)

    BLEND_ALPHA = best_alpha
    cv_oof_proba = BLEND_ALPHA * cv_oof_tfidf + (1.0 - BLEND_ALPHA) * cv_transformer["oof_proba"]
    blend_loss = best_alpha_loss
    print(f"Best transformer blend alpha (tfidf weight): {BLEND_ALPHA:.2f}")

baseline_score = blend_loss
print(f"Selected OOF hamming_loss@0.5: {baseline_score:.6f}")

In [ ]:
# Threshold tuning on selected OOF probabilities (blended if available)
best_thresholds, tuned_score = find_best_thresholds(
    y_true=y,
    y_proba=cv_oof_proba,
    score_fn=hamming_score,
)
print("Best thresholds:", best_thresholds)
print(f"Tuned OOF hamming_loss: {tuned_score:.6f}")

In [ ]:
# Train on full data and infer test (multi-seed LR+SGD TF-IDF + optional transformer blend)
test_proba_lr_list = []
test_proba_sgd_list = []

for seed_value in ENSEMBLE_SEEDS:
    estimator_lr = build_base_classifier(seed=seed_value, C=BEST_C, class_weight=BEST_CLASS_WEIGHT)
    full_model_lr = fit_full_model(x_train_tfidf, y, estimator_lr)
    test_proba_lr_list.append(full_model_lr.predict_proba(x_test_tfidf))

    estimator_sgd = build_sgd_classifier(seed=seed_value, alpha=3e-5, class_weight=BEST_CLASS_WEIGHT)
    full_model_sgd = fit_full_model(x_train_tfidf, y, estimator_sgd)
    test_proba_sgd_list.append(full_model_sgd.predict_proba(x_test_tfidf))

mean_test_proba_lr = np.mean(test_proba_lr_list, axis=0)
mean_test_proba_sgd = np.mean(test_proba_sgd_list, axis=0)
test_proba_tfidf = MODEL_BLEND_WEIGHT * mean_test_proba_lr + (1.0 - MODEL_BLEND_WEIGHT) * mean_test_proba_sgd

test_proba = test_proba_tfidf

if HAS_TRANSFORMER and BLEND_ALPHA < 1.0:
    estimator_tr = build_base_classifier(seed=SEED, C=BEST_C, class_weight=BEST_CLASS_WEIGHT)
    full_model_transformer = fit_full_model(x_train_transformer, y, estimator_tr)
    test_proba_transformer = full_model_transformer.predict_proba(x_test_transformer)
    test_proba = BLEND_ALPHA * test_proba_tfidf + (1.0 - BLEND_ALPHA) * test_proba_transformer

test_pred = apply_thresholds(test_proba, best_thresholds)

submission = sample_submission.copy()
submission["id"] = test_df["id"].values
submission["target"] = format_submission_targets(test_pred)

if str(PROJECT_ROOT).startswith("/kaggle/input"):
    output_path = Path("/kaggle/working/sample_submission.csv")
else:
    output_path = PROJECT_ROOT / "sample_submission.csv"

submission.to_csv(output_path, index=False)
print("Saved submission:", output_path)
display(submission.head())

In [ ]:
print("=== Reproducibility report ===")
print("Seed:", SEED)
print("CV strategy:", CV_STRATEGY)
print("Ensemble seeds:", ENSEMBLE_SEEDS)
print("LR seed losses@0.5:", [round(v, 6) for v in lr_seed_losses])
print("SGD seed losses@0.5:", [round(v, 6) for v in sgd_seed_losses])
print("TF-IDF model blend weight (LR):", MODEL_BLEND_WEIGHT)
print("Selected text variant:", best_variant_name)
print("Transformer available:", HAS_TRANSFORMER)
print("Transformer blend alpha (tfidf weight):", BLEND_ALPHA)
print("Selected C:", BEST_C)
print("Selected class_weight:", BEST_CLASS_WEIGHT)
print("OOF selected loss@0.5:", round(baseline_score, 6))
print("OOF tuned loss:", round(tuned_score, 6))
if transformer_loss is not None:
    print("OOF transformer-only loss@0.5:", round(transformer_loss, 6))
print("Train rows:", len(train_df), "| Test rows:", len(test_df))
if ABLATION_ENABLED and not ablation_df.empty:
    print("Hyperparameter runs:", len(ablation_df))
if 'text_ablation_df' in globals() and not text_ablation_df.empty:
    print("Text-fusion runs:", len(text_ablation_df))